<a href="https://colab.research.google.com/github/jdvelasq/datalabs/blob/master/labs/analisis_de_sentimientos_en_amazon_usando_bayes.ipynb" target="_parent"> <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LAB: Análisis de sentimientos de mensajes en Amazon usando Naive Bayes
===

El archivo que se encuentra disponible en el link

https://raw.githubusercontent.com/jdvelasq/datalabs/master/datasets/amazon_cells_labelled.tsv

contiene mensajes escritos por los usuarios para productos comprados en Amazon y su valoración (positiva, negativa e indeterminada). En este laboratorio se debe construir un clasificador bayesiano que debe ser entrenado con los mensajes valorados, el cual debe ser posteriormente utilizado para valorar los mensajes con valoración indeterminada.

In [1]:
#
# Cargue el archivo usando Pandas e imprima la cantidad de
# registros
#
# Rta/
# 14609
#
import pandas as pd

df = pd.read_csv(
    "https://raw.githubusercontent.com/jdvelasq/datalabs/master/datasets/amazon_cells_labelled.tsv",
    sep="\t",
    names=["text", "value"],
)

len(df)

14609

In [2]:
#
# Imprima el primer mensaje de texto
#
df.text[0]

'I try not to adjust the volume setting to avoid that I turn off the call button which is situated just below the volume adjustment knob.'

In [3]:
#
# Imprima la cantidad de mensajes con NaN
#
# Rta/
# 13609
#
df.value.isna().sum()

13609

In [4]:
#
# Imprima la cantidad de mensajes con valoración igual a 1.0
#
# Rta/
# 500
#
len(df.value[df.value == 1.0])

500

In [5]:
#
# Imprima la cantidad de mensajes con valoración igual a 0.0
#
# Rta/
# 500
#
len(df.value[df.value == 0.0])

500

In [6]:
#
# Genere un nuevo dataset que contenga únicamente los registros
# con valoración positiva o negativa e imprima su longitud
#
# Rta/
# 1000
#
df_data = df.dropna().copy()
len(df_data)

1000

In [7]:
#
# Genere una nueva columna en el nuevo dataset computada como
# el resultado de aplicar el stemmer de Porter al mensaje e
# imprima el primer mensaje transformado
#
# Rta/
# 'So there is no way for me to plug it in here in the US unless I go by a converter.'
#
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()
df_data["stemmed"] = df_data.text.apply(lambda x: " ".join([stemmer.stem(w) for w in x.split()]))
df_data.stemmed[df_data.index[0]]

'So there is no way for me to plug it in here in the US unless I go by a converter.'

In [8]:
#
# Construya la matriz de terminos del documento considerando
# las palabras que tengan una frecuencia entre el 0.1% y el 98%,
# y que esten unicamente conformadas por letras.
#
# Imprima el tamaño del vocabulario.
#
# Rta/
# 1497
#
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer(
    analyzer="word",  # a nivel de palabra
    lowercase=True,  # convierte a minúsculas
    stop_words="english",  # stop_words en inglés
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",  # patrones a reconocer
    binary=True,  # Los valores distintos de cero son fijados en 1
    max_df=0.98,  #  máxima frecuencia a considerar
    min_df=0.001,  # ignora palabras con baja frecuencia
)

dtm = count_vect.fit_transform(df_data.stemmed)

vocabulary = count_vect.get_feature_names()
len(vocabulary)

1497

In [9]:
#
# Construya un clasificador bayesiano que use los primeros
# 500 patrones para entrenamiento y los últimos 500 para
# prueba, e imprima el porcentaje de datos para cada clase
# para la muestra de entrenamiento-
#
# Rta/
# 1.0    52.2
# 0.0    47.8
# Name: value, dtype: float64
#
X_train = dtm[
    :500,
]
X_test = dtm[
    500:,
]

y_train_true = df_data.value[:500]
y_test_true = df_data.value[500:]

round(100 * y_train_true.value_counts() / sum(y_train_true.value_counts()), 2)

1.0    52.2
0.0    47.8
Name: value, dtype: float64

In [10]:
#
# Imprima el porcentaje de datos para cada clase para la muestra
# de prueba
#
# Rta/
# 0.0    52.2
# 1.0    47.8
# Name: value, dtype: float64
#
round(100 * y_test_true.value_counts() / sum(y_test_true.value_counts()), 2)

0.0    52.2
1.0    47.8
Name: value, dtype: float64

In [11]:
#
# Cree un clasificador de Bayes y entrenelo. Realice el pronostico
# para la muestra de entrenamiento y compute la matriz de confusion
#
# Rta/
# array([[214,  25],
#        [  1, 260]])
#
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import confusion_matrix

clf = BernoulliNB()
clf.fit(X_train.toarray(), y_train_true)
y_train_pred = clf.predict(X_train.toarray())

confusion_matrix(y_true=y_train_true, y_pred=y_train_pred)

array([[214,  25],
       [  1, 260]])

In [12]:
#
# Realice el pronóstico para la muestra de entrenamiento y compute
# la matriz de confusión
#
# Rta/
# array([[153, 108],
#        [ 29, 210]])
#
y_test_pred = clf.predict(X_test.toarray())
confusion_matrix(y_true=y_test_true, y_pred=y_test_pred)

array([[153, 108],
       [ 29, 210]])

In [13]:
#
# Realice el pronostico para los mensajes con valoración 
# indeterminada y compute la cantidad de mensajes positivos
#
# Rta/
# 8233
#
df["stemmed"] = df.text.apply(lambda x: " ".join([stemmer.stem(w) for w in x.split()]))

dtm = count_vect.transform(df.stemmed)
df["predict"] = clf.predict(dtm)
df_aux = df[(df.value != 1.0) & (df.value != 0.0)].copy()
int(df_aux.predict[df_aux.predict > 0].sum())

8233